# 1. Purpose and Safety Warning

This notebook is the single master Colab entry point for the leak-free Stage 2 DAOWOD experiment. It runs the same repository code paths for smoke, mini, and full modes: repository checkout, asset validation, DAOWOD gates, PROB bridge validation, train, predict, official evaluation, grouped metrics, artifact validation, Drive persistence, and resume checks.

Safety rules are intentional: the notebook refuses missing CUDA, wrong commits, dirty source trees, missing assets, split leakage, ambiguous strategies, protocol mismatches, failed validation gates, and completed-round overwrites. It never advances from `SMOKE` to `MINI` or `FULL` automatically, and it never starts training until `CONFIRM_LAUNCH = True` is set in the configuration cell.

# 2. User Configuration

Edit only the next cell. `RUN_MODE` defaults to `SMOKE`; switch it manually to `MINI` or `FULL` only after the smoke run has passed. Keep `CONFIRM_LAUNCH = False` until all validation and the run matrix preview look right.

In [ ]:
# USER-EDITABLE CONFIGURATION CELL ONLY
RUN_MODE = "SMOKE"  # one of: "SMOKE", "MINI", "FULL"
CONFIRM_LAUNCH = False

DAOWOD_GIT_URL = "https://github.com/gubiczam/distribution-aware-owod.git"
DAOWOD_COMMIT = "14fb3902b982ff75a21e2cb9920ad98f0bc5ef51"
PROB_GIT_URL = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "980cf3a796f064dd4c56f573ba10cc755143e116"

DRIVE_ROOT = "/content/drive/MyDrive/DAOWOD"
ASSETS_DIR = f"{DRIVE_ROOT}/assets"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
CACHE_DIR = f"{DRIVE_ROOT}/cache"

# Defaults follow the previous pilot Drive layout. Change here if your Drive differs.
CHECKPOINT_PATH = "/content/drive/MyDrive/results/SOWODB/t1.pth"
DINO_WEIGHTS_PATH = "/content/drive/MyDrive/PROB/models/dino_resnet50_pretrain.pth"
OWOD_STAGE_DIR = "/content/drive/MyDrive/owod_stage"
DATA_ARCHIVE_PATH = f"{ASSETS_DIR}/owod_stage.tar.gz"  # optional fallback if OWOD_STAGE_DIR is absent

ALLOW_INSTALL = True
ALLOW_RESUME = True
ALLOW_COMPLETED_SKIP = True
FORCE_RERUN = False
REQUIRE_T4_OR_BETTER = True
ALLOW_NON_T4_GPU = False
RUN_PROPOSAL_EXPORT_SMOKE = True

EXPECTED_CHECKPOINT_SHA256 = "dba5390bffdfdf63058a995f241696df8d06b7fb859aecc8292d9ea02d459a22"
EXPECTED_EVAL_SPLIT_SHA256 = "f58a4a97a8c4c84af337e6ab8dfb4ec97b5d96c6269a601d4f3d4dc3bddef49d"
EXPECTED_CANDIDATE_SHA256 = "70fa185514dcbbba8397781d85275362c888e6ea0c4d6c1325ad6c82fa18aac6"
EXPECTED_REFERENCE_SHA256 = "25a1b33614bcb77c8ef9b238ab878950b62861d0fc048fc58574c7fd0c6df762"

STAGE2_STRATEGY_CONFIGS = [
    "configs/stage2_v2_random.yaml",
    "configs/stage2_v2_uncertainty_objectness_weighted_entropy.yaml",
    "configs/stage2_v2_full.yaml",
    "configs/stage2_v2_full_no_novelty.yaml",
]
SMOKE_CONFIG = "configs/smoke_stage2_t4.yaml"
STAGE1B_CANDIDATE_SPLIT = "outputs/stage1b/stage1b_candidate_500.txt"
STAGE1B_REFERENCE_SPLIT = "outputs/stage1b/stage1b_reference_3500.txt"
STAGE2_CLASS_GROUPS = "outputs/stage2_plan/stage2_class_groups.csv"
EVALUATION_SPLIT_NAME = "owdetr_test"

MODE_OUTPUT_DIRS = {
    "SMOKE": "smoke_stage2",
    "MINI": "mini_stage2",
    "FULL": "full_stage2",
}

In [ ]:
# Notebook helpers. Do not edit unless you are changing the notebook itself.
import csv
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import textwrap
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any

GATE_RESULTS: list[dict[str, Any]] = []
NOTEBOOK_STARTED_AT = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def fail(message: str) -> None:
    raise RuntimeError(message)


def ensure(condition: bool, message: str) -> None:
    if not condition:
        fail(message)


def run_cmd(cmd: list[str], *, cwd: str | Path | None = None, env: dict[str, str] | None = None,
            timeout: int | None = None, gate: str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    start = time.time()
    display = " ".join(str(part) for part in cmd)
    print(f"$ {display}")
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    runtime = time.time() - start
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    status = "PASS" if result.returncode == 0 else "FAIL"
    if gate:
        GATE_RESULTS.append({
            "gate": gate,
            "status": status,
            "runtime_seconds": round(runtime, 3),
            "details": display,
            "returncode": result.returncode,
        })
    if check and result.returncode != 0:
        fail(f"Command failed ({result.returncode}): {display}")
    return result


def stream_cmd(cmd: list[str], *, cwd: str | Path, env: dict[str, str], log_path: Path,
               progress_root: Path, timeout: int | None = None) -> subprocess.CompletedProcess:
    display = " ".join(str(part) for part in cmd)
    print(f"$ {display}")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    start = time.time()
    with log_path.open("a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())} =====\n")
        log.write(display + "\n")
        process = subprocess.Popen(
            [str(part) for part in cmd],
            cwd=str(cwd),
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        last_progress = 0.0
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            now = time.time()
            if now - last_progress >= 60:
                print_progress(progress_root)
                last_progress = now
            if timeout and now - start > timeout:
                process.kill()
                fail(f"Command timed out after {timeout}s: {display}")
        returncode = process.wait()
    if returncode != 0:
        fail(f"Command failed ({returncode}): {display}. See {log_path}")
    return subprocess.CompletedProcess(cmd, returncode)


def sha256_file(path: str | Path) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_ids(path: str | Path) -> list[str]:
    return [line.split()[0] for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def file_info(path: str | Path, expected_sha256: str | None = None) -> dict[str, Any]:
    path = Path(path)
    info: dict[str, Any] = {"path": str(path), "exists": path.exists(), "size": 0, "sha256": None, "expected_sha256": expected_sha256, "status": "FAIL"}
    if path.exists():
        info["size"] = path.stat().st_size if path.is_file() else 0
        if path.is_file() and info["size"] > 0:
            info["sha256"] = sha256_file(path)
        info["status"] = "PASS" if expected_sha256 in (None, info["sha256"]) else "FAIL"
    print(json.dumps(info, indent=2))
    if info["status"] != "PASS":
        fail(f"Asset validation failed: {path}")
    return info


def write_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")


def write_csv(path: str | Path, rows: list[dict[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fields: list[str] = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def git_head(path: str | Path) -> str:
    return run_cmd(["git", "rev-parse", "HEAD"], cwd=path).stdout.strip()


def git_status(path: str | Path) -> str:
    return run_cmd(["git", "status", "--short"], cwd=path).stdout.strip()


def print_gate_table() -> None:
    if not GATE_RESULTS:
        print("No gates recorded yet.")
        return
    print("| gate | status | runtime | details |")
    print("| --- | --- | ---: | --- |")
    for row in GATE_RESULTS:
        print(f"| {row['gate']} | {row['status']} | {row['runtime_seconds']} | `{row['details']}` |")


def print_progress(root: str | Path) -> None:
    root = Path(root)
    manifests = sorted(root.rglob("round_manifest.json")) if root.exists() else []
    completed = 0
    rows = []
    for manifest_path in manifests:
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if manifest.get("completed") is True:
            completed += 1
        rows.append({
            "strategy": manifest.get("strategy"),
            "seed": manifest.get("seed"),
            "round": manifest.get("round_index"),
            "completed": manifest.get("completed"),
            "dir": str(manifest_path.parent),
        })
    print(f"Progress: {completed}/{len(manifests)} manifest(s) completed under {root}")
    for row in rows[-8:]:
        print(row)

# 3. Google Drive Mount

In [ ]:
try:
    from google.colab import drive
except ModuleNotFoundError:
    print("Not running inside Colab; assuming Drive paths are already mounted for local dry validation.")
else:
    drive.mount("/content/drive")

for directory in (DRIVE_ROOT, ASSETS_DIR, RESULTS_DIR, CACHE_DIR):
    Path(directory).mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

# 4. Environment and GPU Validation

In [ ]:
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
except Exception as exc:
    fail(f"PyTorch import failed before setup: {exc}")
print("PyTorch before setup:", torch.__version__)
ensure(torch.cuda.is_available(), "CUDA GPU is required. Runtime > Change runtime type > GPU.")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
acceptable_tokens = ("T4", "L4", "A100", "V100", "P100", "H100")
if REQUIRE_T4_OR_BETTER and not any(token in GPU_NAME.upper() for token in acceptable_tokens):
    ensure(ALLOW_NON_T4_GPU, f"GPU {GPU_NAME!r} is not T4-or-better by policy. Set ALLOW_NON_T4_GPU=True only deliberately.")
run_cmd(["nvidia-smi"], gate="gpu_nvidia_smi")

# 5. Path Resolution

In [ ]:
RUN_MODE = RUN_MODE.upper().strip()
ensure(RUN_MODE in {"SMOKE", "MINI", "FULL"}, "RUN_MODE must be SMOKE, MINI, or FULL.")

DAOWOD_REPO = Path("/content/distribution-aware-owod")
PROB_REPO = Path("/content/PROB")
RUNTIME_ROOT = Path("/content/daowod_runtime") / RUN_MODE.lower()
LOCAL_OUTPUT_ROOT = Path("/content/daowod_runs") / RUN_MODE.lower()
MODE_RESULTS_DIR = Path(RESULTS_DIR) / MODE_OUTPUT_DIRS[RUN_MODE]
LOG_DIR = MODE_RESULTS_DIR / "logs"
LOCAL_STAGE = Path("/content/owod_stage")
COLAB_USER_ROOT = Path("/Users/gubiczam")
COLAB_PROB_LINK = COLAB_USER_ROOT / "Documents" / "PROB"
COLAB_STAGE_LINK = COLAB_USER_ROOT / "owod_stage"
COLAB_RESULTS_LINK_DIR = COLAB_USER_ROOT / "Downloads" / "results" / "SOWODB"

for directory in (RUNTIME_ROOT, LOCAL_OUTPUT_ROOT, MODE_RESULTS_DIR, LOG_DIR, COLAB_PROB_LINK.parent, COLAB_RESULTS_LINK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "run_mode": RUN_MODE,
    "daowod_repo": str(DAOWOD_REPO),
    "prob_repo": str(PROB_REPO),
    "runtime_root": str(RUNTIME_ROOT),
    "local_output_root": str(LOCAL_OUTPUT_ROOT),
    "mode_results_dir": str(MODE_RESULTS_DIR),
}, indent=2))

# 6. Repository Checkout

In [ ]:
def safe_checkout(url: str, commit: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        try:
            current_head = git_head(target)
            current_status = git_status(target)
            if current_head == commit and current_status == "":
                print(f"Using existing clean checkout: {target} @ {commit}")
                return
            backup = target.with_name(f"{target.name}.backup.{int(time.time())}")
            print(f"Existing checkout is not the requested clean commit; moving it to {backup}")
            shutil.move(str(target), str(backup))
        except Exception:
            backup = target.with_name(f"{target.name}.unreadable.{int(time.time())}")
            print(f"Existing path is not a usable checkout; moving it to {backup}")
            shutil.move(str(target), str(backup))
    tmp = target.with_name(f"{target.name}.tmp.{int(time.time())}")
    run_cmd(["git", "clone", url, str(tmp)], timeout=900)
    run_cmd(["git", "fetch", "origin", commit], cwd=tmp, timeout=300)
    run_cmd(["git", "checkout", "--detach", commit], cwd=tmp, timeout=300)
    ensure(git_head(tmp) == commit, f"Checkout failed for {target}")
    ensure(git_status(tmp) == "", f"Fresh checkout is dirty: {tmp}")
    shutil.move(str(tmp), str(target))

safe_checkout(DAOWOD_GIT_URL, DAOWOD_COMMIT, DAOWOD_REPO)
safe_checkout(PROB_GIT_URL, PROB_COMMIT, PROB_REPO)

# 7. Commit Validation

In [ ]:
for name, repo, expected in (("DAOWOD", DAOWOD_REPO, DAOWOD_COMMIT), ("PROB", PROB_REPO, PROB_COMMIT)):
    observed = git_head(repo)
    status = git_status(repo)
    print(f"{name} HEAD: {observed}")
    ensure(observed == expected, f"{name} commit mismatch: {observed} != {expected}")
    ensure(status == "", f"{name} source tree is dirty before runtime assets/builds:\n{status}")

required_repo_paths = [
    SMOKE_CONFIG,
    *STAGE2_STRATEGY_CONFIGS,
    STAGE1B_CANDIDATE_SPLIT,
    STAGE1B_REFERENCE_SPLIT,
    STAGE2_CLASS_GROUPS,
    "src/daowod/config.py",
    "src/daowod/experiment.py",
]
for rel in required_repo_paths:
    ensure((DAOWOD_REPO / rel).exists(), f"DAOWOD checkout is missing required Stage 1B/Stage 2 file: {rel}. Commit or update DAOWOD_COMMIT before running.")
print("Commit validation PASS")

# 8. Asset Validation

In [ ]:
def extract_archive_if_needed(archive: Path, destination: Path) -> None:
    ensure(archive.exists(), f"Dataset directory {destination} is absent and DATA_ARCHIVE_PATH is missing: {archive}")
    digest = sha256_file(archive)
    marker = destination / ".daowod_extract_complete.json"
    if destination.exists() and marker.exists():
        payload = json.loads(marker.read_text(encoding="utf-8"))
        if payload.get("source_sha256") == digest:
            print(f"Using cached extraction: {destination}")
            return
    tmp = destination.with_name(destination.name + f".tmp.{int(time.time())}")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    if archive.suffix == ".zip":
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(tmp)
    elif archive.suffixes[-2:] == [".tar", ".gz"] or archive.suffix in {".tar", ".tgz"}:
        with tarfile.open(archive) as tf:
            tf.extractall(tmp)
    else:
        fail(f"Unsupported dataset archive format: {archive}")
    marker_payload = {"source": str(archive), "source_sha256": digest, "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    write_json(tmp / ".daowod_extract_complete.json", marker_payload)
    if destination.exists():
        backup = destination.with_name(destination.name + f".backup.{int(time.time())}")
        shutil.move(str(destination), str(backup))
    shutil.move(str(tmp), str(destination))

checkpoint = Path(CHECKPOINT_PATH)
dino = Path(DINO_WEIGHTS_PATH)
owod_stage_drive = Path(OWOD_STAGE_DIR)
file_info(checkpoint, EXPECTED_CHECKPOINT_SHA256)
file_info(dino, None)

if owod_stage_drive.exists():
    print("Using OWOD_STAGE_DIR:", owod_stage_drive)
    if LOCAL_STAGE.exists() or LOCAL_STAGE.is_symlink():
        LOCAL_STAGE.unlink() if LOCAL_STAGE.is_symlink() else None
    if not LOCAL_STAGE.exists():
        LOCAL_STAGE.symlink_to(owod_stage_drive, target_is_directory=True)
else:
    extract_archive_if_needed(Path(DATA_ARCHIVE_PATH), Path(CACHE_DIR) / "owod_stage")
    if LOCAL_STAGE.exists() or LOCAL_STAGE.is_symlink():
        LOCAL_STAGE.unlink() if LOCAL_STAGE.is_symlink() else None
    if not LOCAL_STAGE.exists():
        LOCAL_STAGE.symlink_to(Path(CACHE_DIR) / "owod_stage", target_is_directory=True)

# Compatibility links for the checked-in Stage 2 YAML paths.
if COLAB_PROB_LINK.exists() or COLAB_PROB_LINK.is_symlink():
    COLAB_PROB_LINK.unlink() if COLAB_PROB_LINK.is_symlink() else None
if not COLAB_PROB_LINK.exists():
    COLAB_PROB_LINK.symlink_to(PROB_REPO, target_is_directory=True)
if COLAB_STAGE_LINK.exists() or COLAB_STAGE_LINK.is_symlink():
    COLAB_STAGE_LINK.unlink() if COLAB_STAGE_LINK.is_symlink() else None
if not COLAB_STAGE_LINK.exists():
    COLAB_STAGE_LINK.symlink_to(LOCAL_STAGE, target_is_directory=True)
checkpoint_link = COLAB_RESULTS_LINK_DIR / "t1.pth"
if checkpoint_link.exists() or checkpoint_link.is_symlink():
    checkpoint_link.unlink()
checkpoint_link.symlink_to(checkpoint)

# PROB loads DINO weights through a relative asset path. This places an asset symlink, not a source patch.
prob_dino_path = PROB_REPO / "models" / "dino_resnet50_pretrain.pth"
if prob_dino_path.exists() or prob_dino_path.is_symlink():
    prob_dino_path.unlink()
prob_dino_path.symlink_to(dino)

critical_assets = [
    LOCAL_STAGE / "ImageSets" / "OWDETR" / f"{EVALUATION_SPLIT_NAME}.txt",
    LOCAL_STAGE / "Annotations",
    LOCAL_STAGE / "JPEGImages",
    checkpoint_link,
    prob_dino_path,
]
for asset in critical_assets:
    ensure(asset.exists(), f"Missing critical asset: {asset}")
file_info(LOCAL_STAGE / "ImageSets" / "OWDETR" / f"{EVALUATION_SPLIT_NAME}.txt", EXPECTED_EVAL_SPLIT_SHA256)
print("Asset validation PASS")

# 9. Dataset and Split Validation

In [ ]:
candidate_path = DAOWOD_REPO / STAGE1B_CANDIDATE_SPLIT
reference_path = DAOWOD_REPO / STAGE1B_REFERENCE_SPLIT
eval_path = LOCAL_STAGE / "ImageSets" / "OWDETR" / f"{EVALUATION_SPLIT_NAME}.txt"
class_groups_path = DAOWOD_REPO / STAGE2_CLASS_GROUPS
file_info(candidate_path, EXPECTED_CANDIDATE_SHA256)
file_info(reference_path, EXPECTED_REFERENCE_SHA256)
file_info(class_groups_path, None)

candidate_ids = read_ids(candidate_path)
reference_ids = read_ids(reference_path)
eval_ids = read_ids(eval_path)
ensure(len(candidate_ids) == 500, f"Expected 500 candidate IDs, got {len(candidate_ids)}")
ensure(len(reference_ids) == 3500, f"Expected 3500 reference IDs, got {len(reference_ids)}")
ensure(len(set(candidate_ids)) == len(candidate_ids), "Candidate split has duplicates")
ensure(len(set(reference_ids)) == len(reference_ids), "Reference split has duplicates")
checks = {
    "candidate_reference_overlap": len(set(candidate_ids) & set(reference_ids)),
    "candidate_eval_overlap": len(set(candidate_ids) & set(eval_ids)),
    "reference_eval_overlap": len(set(reference_ids) & set(eval_ids)),
}
print(json.dumps(checks, indent=2))
ensure(all(value == 0 for value in checks.values()), f"Split leakage detected: {checks}")

missing_xml = [image_id for image_id in candidate_ids[:20] + reference_ids[:20] + eval_ids[:20] if not (LOCAL_STAGE / "Annotations" / f"{image_id}.xml").exists()]
missing_jpg = [image_id for image_id in candidate_ids[:20] + reference_ids[:20] + eval_ids[:20] if not (LOCAL_STAGE / "JPEGImages" / f"{image_id}.jpg").exists()]
ensure(not missing_xml, f"Missing sampled XML annotations: {missing_xml[:10]}")
ensure(not missing_jpg, f"Missing sampled JPEG images: {missing_jpg[:10]}")
print("Dataset and split validation PASS")

# 10. Dependency Installation

In [ ]:
ensure(ALLOW_INSTALL, "ALLOW_INSTALL is False; refusing dependency installation.")
print("Python before install:", sys.version)
try:
    import torch
    print("PyTorch before install:", torch.__version__)
except Exception as exc:
    print("PyTorch before install unavailable:", exc)

run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip"], gate="pip_upgrade", timeout=600)
run_cmd([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=DAOWOD_REPO, gate="install_daowod", timeout=900)

prob_venv_python = PROB_REPO / ".venv" / "bin" / "python"
if not prob_venv_python.exists():
    run_cmd([sys.executable, "-m", "venv", str(PROB_REPO / ".venv")], gate="create_prob_venv", timeout=300)
run_cmd([str(prob_venv_python), "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], gate="prob_pip_upgrade", timeout=900)
# Colab-compatible dependency set: keeps PROB's required libraries but avoids old pandas/scikit-image/notebook build pins.
prob_requirements = RUNTIME_ROOT / "prob_colab_requirements.txt"
prob_requirements.write_text("\n".join([
    "wandb==0.13.4",
    "einops==0.5.0",
    "pycocotools>=2.0.7,<3",
    "scikit-image>=0.22,<1",
    "joblib>=1.2,<2",
    "tqdm>=4.64,<5",
    "pandas>=2.2,<3",
    "seaborn>=0.12,<1",
    "ipdb>=0.13,<1",
]) + "\n", encoding="utf-8")
run_cmd([str(prob_venv_python), "-m", "pip", "install", "-r", str(prob_requirements)], gate="install_prob_dependencies", timeout=1800)

run_cmd([sys.executable, "-m", "pip", "freeze"], cwd=DAOWOD_REPO, gate="freeze_daowod", timeout=300)
prob_freeze = run_cmd([str(prob_venv_python), "-m", "pip", "freeze"], cwd=PROB_REPO, gate="freeze_prob", timeout=300)
write_json(MODE_RESULTS_DIR / "environment_report.json", {
    "started_at_utc": NOTEBOOK_STARTED_AT,
    "python": sys.version,
    "platform": platform.platform(),
    "run_mode": RUN_MODE,
    "daowod_commit": DAOWOD_COMMIT,
    "prob_commit": PROB_COMMIT,
    "prob_freeze": prob_freeze.stdout.splitlines(),
})
import torch
print("PyTorch after setup:", torch.__version__)

# 11. CUDA Extension Build

In [ ]:
prob_venv_python = PROB_REPO / ".venv" / "bin" / "python"
ops_dir = PROB_REPO / "models" / "ops"
build_log = LOG_DIR / "cuda_extension_build.log"
start = time.time()
try:
    result = run_cmd(["bash", "make.sh"], cwd=ops_dir, gate="build_deformable_detr_cuda_extension", timeout=1800, check=False)
    build_log.write_text((result.stdout or "") + "\nSTDERR\n" + (result.stderr or ""), encoding="utf-8")
    ensure(result.returncode == 0, f"CUDA extension build failed; see {build_log}")
    test_result = run_cmd([str(prob_venv_python), "test.py"], cwd=ops_dir, gate="test_deformable_detr_cuda_extension", timeout=900, check=False)
    ensure(test_result.returncode == 0, "CUDA extension test failed")
finally:
    print(f"CUDA build section runtime: {time.time() - start:.1f}s")

# 12. DAOWOD Validation Gates

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(DAOWOD_REPO / "src")
env["DAOWOD_RESUME_COMPLETED"] = "1" if ALLOW_RESUME else "0"
run_cmd([sys.executable, "-m", "compileall", "src", "analysis", "tests"], cwd=DAOWOD_REPO, env=env, gate="compileall", timeout=600)
run_cmd(["ruff", "format", "--check", "."], cwd=DAOWOD_REPO, env=env, gate="ruff_format_check", timeout=300)
run_cmd(["ruff", "check", "."], cwd=DAOWOD_REPO, env=env, gate="ruff_check", timeout=300)
run_cmd([sys.executable, "-m", "pytest"], cwd=DAOWOD_REPO, env=env, gate="pytest", timeout=1200)
run_cmd([sys.executable, "-m", "daowod.cli", "strategies", "--required-only"], cwd=DAOWOD_REPO, env=env, gate="strategy_registry", timeout=300)
run_cmd([str(prob_venv_python), "daowod_prob_bridge.py", "check"], cwd=PROB_REPO, gate="prob_bridge_check", timeout=300)

checkpoint_load_code = """
import torch, sys
path=sys.argv[1]
ckpt=torch.load(path, map_location='cpu')
assert isinstance(ckpt, dict), type(ckpt)
assert 'model' in ckpt or len(ckpt) > 0
print('checkpoint keys', sorted(list(ckpt.keys()))[:20])
"""
run_cmd([str(prob_venv_python), "-c", checkpoint_load_code, str(checkpoint_link)], cwd=PROB_REPO, gate="checkpoint_load", timeout=600)

if RUN_PROPOSAL_EXPORT_SMOKE:
    one_id_file = RUNTIME_ROOT / "proposal_smoke_one_id.txt"
    one_id_file.write_text(candidate_ids[0] + "\n", encoding="utf-8")
    proposal_smoke_output = RUNTIME_ROOT / "proposal_smoke.npz"
    run_cmd([
        str(prob_venv_python), "daowod_prob_bridge.py", "predict",
        "--image-ids", str(one_id_file),
        "--checkpoint", str(checkpoint_link),
        "--output", str(proposal_smoke_output),
        "--data-root", str(LOCAL_STAGE),
        "--dataset", "OWDETR",
        "--prev-introduced-classes", "0",
        "--current-introduced-classes", "19",
        "--num-classes", "81",
        "--objectness-temperature", "1",
        "--device", "cuda",
        "--max-proposals-per-image", "100",
    ], cwd=PROB_REPO, gate="proposal_export_smoke", timeout=1800)
    file_info(proposal_smoke_output, None)
print_gate_table()

# 13. Protocol/Config Validation

In [ ]:
import yaml

MODE_CONFIGS: list[Path] = []
if RUN_MODE == "SMOKE":
    source_configs = [SMOKE_CONFIG]
elif RUN_MODE in {"MINI", "FULL"}:
    source_configs = STAGE2_STRATEGY_CONFIGS
else:
    fail(f"Unknown RUN_MODE {RUN_MODE}")

for rel in source_configs:
    source = DAOWOD_REPO / rel
    cfg = yaml.safe_load(source.read_text(encoding="utf-8"))
    cfg["output_dir"] = str(LOCAL_OUTPUT_ROOT / Path(rel).stem)
    cfg["prob"]["repository_path"] = str(PROB_REPO)
    cfg["prob"]["initial_checkpoint"] = str(checkpoint_link)
    cfg["protocol"]["checkpoint"] = str(checkpoint_link)
    cfg["protocol"]["data_root"] = str(LOCAL_STAGE)
    cfg["dataset"]["annotations_dir"] = str(LOCAL_STAGE / "Annotations")
    if RUN_MODE == "MINI":
        cfg["active_learning"]["seeds"] = [0]
    if RUN_MODE == "SMOKE":
        cfg["active_learning"]["seeds"] = [0]
    dest = RUNTIME_ROOT / Path(rel).name
    dest.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    MODE_CONFIGS.append(dest)

for cfg_path in MODE_CONFIGS:
    manifest_path = RUNTIME_ROOT / f"{cfg_path.stem}_validate_manifest.json"
    run_cmd([sys.executable, "-m", "daowod.cli", "validate", "--config", str(cfg_path), "--manifest", str(manifest_path)], cwd=DAOWOD_REPO, env=env, gate=f"config_validate:{cfg_path.name}", timeout=300)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    protocol = manifest["config"]["protocol"]
    ensure(protocol["evaluation_split_sha256"] == EXPECTED_EVAL_SPLIT_SHA256, "Evaluation split digest mismatch in config manifest")
    ensure(protocol["allow_candidate_evaluation_overlap"] is False, "Leakage override unexpectedly enabled")
print_gate_table()

# 14. Run Matrix Preview

In [ ]:
def config_matrix(config_paths: list[Path]) -> list[dict[str, Any]]:
    rows = []
    for cfg_path in config_paths:
        cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
        strategies = cfg["acquisition"].get("strategies") or [cfg["active_learning"]["strategy"]]
        seeds = cfg["active_learning"]["seeds"]
        rounds = int(cfg["active_learning"]["rounds"])
        for strategy in strategies:
            for seed in seeds:
                for round_index in range(1, rounds + 1):
                    rows.append({
                        "config": cfg_path.name,
                        "strategy": strategy,
                        "seed": seed,
                        "round": round_index,
                        "budget": cfg["active_learning"]["budget_per_round"],
                        "output_root": cfg["output_dir"],
                    })
    return rows

RUN_MATRIX = config_matrix(MODE_CONFIGS)
train_calls = len(RUN_MATRIX)
non_random_calls = sum(1 for row in RUN_MATRIX if "random" not in row["strategy"])
predict_calls = non_random_calls * 2
evaluate_calls = train_calls
runtime_estimate_hours = {"SMOKE": 0.75, "MINI": 21.0, "FULL": 63.0}[RUN_MODE]
storage_estimate_gb = {"SMOKE": 2.0, "MINI": 18.0, "FULL": 52.8}[RUN_MODE]
preview = {
    "mode": RUN_MODE,
    "configs": [str(path) for path in MODE_CONFIGS],
    "strategies": sorted(set(row["strategy"] for row in RUN_MATRIX)),
    "seeds": sorted(set(row["seed"] for row in RUN_MATRIX)),
    "rounds": sorted(set(row["round"] for row in RUN_MATRIX)),
    "train_calls": train_calls,
    "predict_calls": predict_calls,
    "evaluate_calls": evaluate_calls,
    "expected_runtime_hours": runtime_estimate_hours,
    "estimated_storage_gb": storage_estimate_gb,
    "local_output_root": str(LOCAL_OUTPUT_ROOT),
    "drive_output_root": str(MODE_RESULTS_DIR),
}
print(json.dumps(preview, indent=2))
print("| config | strategy | seed | round | budget |")
print("| --- | --- | ---: | ---: | ---: |")
for row in RUN_MATRIX:
    print(f"| {row['config']} | {row['strategy']} | {row['seed']} | {row['round']} | {row['budget']} |")
write_csv(RUNTIME_ROOT / "resolved_run_matrix.csv", RUN_MATRIX)
write_json(RUNTIME_ROOT / "resolved_run_matrix_summary.json", preview)

# 15. Explicit Launch Confirmation

In [ ]:
print(json.dumps(preview, indent=2))
ensure(CONFIRM_LAUNCH is True, "Training/evaluation launch is blocked. Set CONFIRM_LAUNCH = True in the configuration cell after reviewing the run matrix.")
if RUN_MODE in {"MINI", "FULL"}:
    print(f"You explicitly requested {RUN_MODE}. This will not run unless CONFIRM_LAUNCH is True, which it now is.")
ensure(torch.cuda.is_available(), "CUDA disappeared before launch.")

# 16. Experiment Execution

In [ ]:
execution_env = os.environ.copy()
execution_env["PYTHONPATH"] = str(DAOWOD_REPO / "src")
execution_env["DAOWOD_RESUME_COMPLETED"] = "1" if ALLOW_RESUME and ALLOW_COMPLETED_SKIP else "0"
if FORCE_RERUN:
    fail("FORCE_RERUN is intentionally unsupported for completed Stage 2 rounds. Move old outputs manually only after preserving them.")

for cfg_path in MODE_CONFIGS:
    log_path = LOG_DIR / f"{cfg_path.stem}_campaign.log"
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    output_root = Path(cfg["output_dir"])
    print(f"Launching {cfg_path.name} -> {output_root}")
    stream_cmd([sys.executable, "-m", "daowod.cli", "campaign", "--config", str(cfg_path)], cwd=DAOWOD_REPO, env=execution_env, log_path=log_path, progress_root=output_root, timeout=None)
print("Experiment execution cell completed")

# 17. Live Progress Summary

In [ ]:
for cfg_path in MODE_CONFIGS:
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    print_progress(Path(cfg["output_dir"]))

# 18. Artifact Validation

In [ ]:
def validate_round(round_dir: Path) -> list[dict[str, Any]]:
    manifest_path = round_dir / "round_manifest.json"
    ensure(manifest_path.exists(), f"Missing round manifest: {round_dir}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    ensure(manifest.get("completed") is True, f"Round is not completed: {round_dir}")
    ensure(manifest.get("resolved_command_parity", {}).get("status") == "ok", f"Resolved command parity failed: {round_dir}")
    overlaps = manifest.get("overlap_policy", {})
    ensure(not overlaps.get("candidate_evaluation_overlap"), f"Candidate/eval leakage in {round_dir}")
    ensure(not overlaps.get("labelled_evaluation_overlap"), f"Labelled/eval leakage in {round_dir}")
    ensure(manifest.get("evaluation_ids_sha256") == sha256_file(round_dir / "evaluation_ids.txt"), f"Evaluation ID digest mismatch in {round_dir}")
    ensure(sha256_file(LOCAL_STAGE / "ImageSets" / "OWDETR" / f"{EVALUATION_SPLIT_NAME}.txt") == EXPECTED_EVAL_SPLIT_SHA256, "Canonical eval split changed")

    required = [
        "checkpoint.pth",
        "metrics.json",
        "candidate_ids_before_selection.txt",
        "reference_ids.txt",
        "labelled_ids_before_selection.txt",
        "selected_ids.txt",
        "labelled_ids.txt",
        "training_ids.txt",
        "remaining_pool_ids.txt",
        "evaluation_ids.txt",
    ]
    strategy_name = str(manifest.get("strategy", ""))
    if strategy_name != "random":
        required.extend(["candidate_proposals.npz", "reference_proposals.npz", "proposal_scores.csv", "image_scores.csv", "component_diagnostics.json"])
    if manifest.get("grouped_metrics") is not None:
        required.append("grouped_metrics.json")
    rows = []
    for name in required:
        path = round_dir / name
        ensure(path.exists() and path.stat().st_size > 0, f"Missing or empty artifact {path}")
        if path.suffix == ".json":
            json.loads(path.read_text(encoding="utf-8"))
        if path.suffix == ".csv":
            with path.open(newline="", encoding="utf-8") as handle:
                header = next(csv.reader(handle))
            ensure(len(header) > 0, f"CSV has no columns: {path}")
        rows.append({"round_dir": str(round_dir), "artifact": name, "path": str(path), "size": path.stat().st_size, "sha256": sha256_file(path)})
    labelled_before = read_ids(round_dir / "labelled_ids_before_selection.txt")
    selected = read_ids(round_dir / "selected_ids.txt")
    labelled_after = read_ids(round_dir / "labelled_ids.txt")
    ensure(len(labelled_after) == len(labelled_before) + len(selected), f"Labelled set did not grow by selected budget: {round_dir}")
    ensure(len(selected) == len(set(selected)), f"Selected IDs repeat within round: {round_dir}")
    return rows

ARTIFACT_INVENTORY: list[dict[str, Any]] = []
for cfg_path in MODE_CONFIGS:
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    for manifest_path in sorted(Path(cfg["output_dir"]).rglob("round_manifest.json")):
        ARTIFACT_INVENTORY.extend(validate_round(manifest_path.parent))
expected_rounds = len(RUN_MATRIX)
observed_rounds = len({row["round_dir"] for row in ARTIFACT_INVENTORY})
ensure(observed_rounds == expected_rounds, f"Expected {expected_rounds} completed rounds, validated {observed_rounds}")
write_csv(MODE_RESULTS_DIR / "artifact_inventory.csv", ARTIFACT_INVENTORY)
print(f"Artifact validation PASS for {observed_rounds} rounds")

# 19. Persistence Verification

In [ ]:
def tree_digest_rows(root: Path) -> list[dict[str, Any]]:
    rows = []
    for path in sorted(root.rglob("*")):
        if path.is_file():
            rows.append({"relative_path": str(path.relative_to(root)), "size": path.stat().st_size, "sha256": sha256_file(path)})
    return rows

PERSISTENCE_REPORT: list[dict[str, Any]] = []
for cfg_path in MODE_CONFIGS:
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    local_root = Path(cfg["output_dir"])
    drive_root = MODE_RESULTS_DIR / "campaign_outputs" / local_root.name
    local_rows = tree_digest_rows(local_root)
    ensure(local_rows, f"No local artifacts to persist: {local_root}")
    if drive_root.exists():
        drive_rows = tree_digest_rows(drive_root)
        if drive_rows == local_rows and ALLOW_COMPLETED_SKIP:
            print(f"Identical persisted output already exists, skipping copy: {drive_root}")
        else:
            fail(f"Drive output already exists and differs: {drive_root}")
    else:
        tmp = drive_root.with_name(drive_root.name + f".tmp.{int(time.time())}")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(local_root, tmp)
        copied_rows = tree_digest_rows(tmp)
        ensure(copied_rows == local_rows, f"Digest mismatch after Drive copy: {drive_root}")
        tmp.rename(drive_root)
        print(f"Persisted {local_root} -> {drive_root}")
    final_rows = tree_digest_rows(drive_root)
    ensure(final_rows == local_rows, f"Final persisted tree mismatch: {drive_root}")
    PERSISTENCE_REPORT.append({"local_root": str(local_root), "drive_root": str(drive_root), "file_count": len(local_rows), "status": "PASS"})
write_json(MODE_RESULTS_DIR / "persistence_report.json", {"schema": "stage2_colab_persistence_v1", "rows": PERSISTENCE_REPORT})
print("Persistence verification PASS")

# 20. Resume Verification

In [ ]:
RESUME_REPORT = {"mode": RUN_MODE, "status": "SKIPPED", "details": "Resume test is required only for SMOKE."}
if RUN_MODE == "SMOKE":
    cfg_path = MODE_CONFIGS[0]
    log_path = LOG_DIR / f"{cfg_path.stem}_resume_check.log"
    result = run_cmd([sys.executable, "-m", "daowod.cli", "campaign", "--config", str(cfg_path)], cwd=DAOWOD_REPO, env=execution_env, timeout=900, check=False)
    log_path.write_text((result.stdout or "") + "\nSTDERR\n" + (result.stderr or ""), encoding="utf-8")
    if ALLOW_RESUME and ALLOW_COMPLETED_SKIP:
        ensure(result.returncode == 0, "Resume-enabled SMOKE rerun should safely skip completed rounds and exit 0")
        RESUME_REPORT = {"mode": RUN_MODE, "status": "PASS", "details": "Completed round loaded through DAOWOD_RESUME_COMPLETED without extra training."}
    else:
        ensure(result.returncode != 0 and "Completed round" in ((result.stdout or "") + (result.stderr or "")), "Overwrite refusal was not observed")
        RESUME_REPORT = {"mode": RUN_MODE, "status": "PASS", "details": "Overwrite refusal observed."}
write_json(MODE_RESULTS_DIR / "resume_report.json", RESUME_REPORT)
print(json.dumps(RESUME_REPORT, indent=2))

# 21. Final PASS / FAIL Report

In [ ]:
write_csv(MODE_RESULTS_DIR / "gate_results.csv", GATE_RESULTS)
all_gates_pass = all(row["status"] == "PASS" for row in GATE_RESULTS)
all_persistence_pass = all(row["status"] == "PASS" for row in PERSISTENCE_REPORT)
resume_ok = RESUME_REPORT["status"] in {"PASS", "SKIPPED"}
rounds_ok = len({row["round_dir"] for row in ARTIFACT_INVENTORY}) == len(RUN_MATRIX)
if all_gates_pass and all_persistence_pass and resume_ok and rounds_ok:
    FINAL_VERDICT = "PASS"
elif rounds_ok and all_gates_pass:
    FINAL_VERDICT = "CONDITIONAL PASS"
else:
    FINAL_VERDICT = "FAIL"
summary = {
    "schema": "stage2_colab_run_summary_v1",
    "verdict": FINAL_VERDICT,
    "mode": RUN_MODE,
    "started_at_utc": NOTEBOOK_STARTED_AT,
    "finished_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "daowod_commit": DAOWOD_COMMIT,
    "prob_commit": PROB_COMMIT,
    "rounds_expected": len(RUN_MATRIX),
    "rounds_validated": len({row["round_dir"] for row in ARTIFACT_INVENTORY}),
    "gate_results": GATE_RESULTS,
    "persistence": PERSISTENCE_REPORT,
    "resume": RESUME_REPORT,
    "results_dir": str(MODE_RESULTS_DIR),
}
write_json(MODE_RESULTS_DIR / "colab_run_summary.json", summary)
md_lines = [
    "# Stage 2 Colab Run Summary",
    "",
    f"Final verdict: **{FINAL_VERDICT}**",
    f"Mode: `{RUN_MODE}`",
    f"DAOWOD commit: `{DAOWOD_COMMIT}`",
    f"PROB commit: `{PROB_COMMIT}`",
    f"Rounds validated: {summary['rounds_validated']} / {summary['rounds_expected']}",
    f"Results directory: `{MODE_RESULTS_DIR}`",
]
(MODE_RESULTS_DIR / "colab_run_summary.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print("FINAL VERDICT:", FINAL_VERDICT)
print(json.dumps(summary, indent=2, default=str))

# 22. Result Locations and Next Action

Primary result directory is derived from the configuration:

- `SMOKE`: `RESULTS_DIR/smoke_stage2`
- `MINI`: `RESULTS_DIR/mini_stage2`
- `FULL`: `RESULTS_DIR/full_stage2`

After a `SMOKE` PASS, change only the configuration cell: set `RUN_MODE = "MINI"`, review the run matrix, then set `CONFIRM_LAUNCH = True`. After MINI passes, repeat deliberately for `FULL`. The notebook never changes mode for you.

In [ ]:
print("Mode results:", MODE_RESULTS_DIR)
for name in ["colab_run_summary.json", "colab_run_summary.md", "gate_results.csv", "artifact_inventory.csv", "environment_report.json", "persistence_report.json", "resume_report.json"]:
    path = MODE_RESULTS_DIR / name
    print({"path": str(path), "exists": path.exists(), "size": path.stat().st_size if path.exists() else 0})